# IOAI — 2024 On Site Round Lost In Hyperspace — ⭐모범답안 (Colab 자동 설정판)

이 노트북은 IOAI 로컬 연습 사이트에서 **데이터·학습환경이 자동 준비**되도록 생성되었습니다.
아래 **설정 셀을 먼저 실행**하면 공식 GitHub 저장소에서 이 문제 폴더만 부분 클론으로 받아
(전체 6.6GB 가 아니라 해당 폴더만), 그 폴더로 이동한 뒤 이후 셀이 그대로 학습/예측을 합니다.
완료 후 생성되는 제출 파일을 내려받아 연습 사이트의 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** 로 바꾸면 학습이 빨라집니다.

In [ ]:
# === 데이터 + 환경 자동 설정 (가장 먼저 실행) ===
# 공식 공개 저장소에서 이 문제 폴더만 부분 클론(sparse)으로 받고 그 폴더로 이동한다.
import os
REPO_URL = "https://github.com/IOAI-official/IOAI-2024"
CLONE = "IOAI-2024"
SUBDIR = "On-Site-Round/Lost_in_Hyperspace"
WORKDIR = "On-Site-Round/Lost_in_Hyperspace/Solution"
# Colab 은 /content 가 홈. 재실행해도 경로가 안정적이도록 고정 기준에서 시작한다.
BASE = "/content" if os.path.isdir("/content") else os.getcwd()
os.chdir(BASE)
if not os.path.isdir(os.path.join(CLONE, SUBDIR)):
    !git clone --filter=blob:none --no-checkout --depth 1 $REPO_URL $CLONE
    !cd $CLONE && git sparse-checkout set "$SUBDIR"
    !cd $CLONE && git checkout
os.chdir(os.path.join(BASE, CLONE, WORKDIR))
print("작업 폴더:", os.getcwd())
print("내용:", sorted(os.listdir(".")))

<img src="./figs/IOAI-Logo.png" alt="IOAI Logo" width="200" height="auto">

[IOAI 2024 (Burgas, Bulgaria), On-Site Round](https://ioai-official.org/bulgaria-2024)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IOAI-official/IOAI-2024/blob/main/On-Site-Round/Lost_in_Hyperspace/Solutioin/Lost_in_Hyperspace_Solution.ipynb)

# Lost in a Hyperspace: Baseline Solution

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt

SCALING_WEIGHTS = [100/15, 100/8, 100/100]

In [ ]:
data = pd.read_pickle('../training_set/ml_data_onsite_start.pickle')
for key in data.keys():
  print(key)

In [ ]:
for key in data['X'].keys():
  print(key)

In [ ]:
for key in data['y'].keys():
  print(key)

In [ ]:
X_train = data['X']['train']
y_train = data['y']['train']

X_val = data['X']['val']
y_val = data['y']['val']

X_test = data['X']['live_test']

In [ ]:
X_train.shape, y_train.shape, X_val.shape, y_val.shape, X_test.shape

In [ ]:
def vis(arr):
  plt.figure(figsize=(8, 8))

  cnt = 1
  for z in range(5):
    for q in range(6):
      plt.subplot(5, 6, cnt)
      plt.imshow(arr[:, :, z, q], vmin=-40, vmax=40, cmap='hsv')
      plt.grid()
      plt.axis('off')
      cnt += 1
  plt.tight_layout()

In [ ]:
vis(X_train[0])

## Functions for result evaluation / writing predictions

Do not change it!

In [ ]:
def test_solution(X_train, y_train, X_val, y_val, feature_num=0):
    assert X_train.shape[-1] <= 300, "Too many features! Should be less than 300"
    assert X_val.shape[-1] <= 300, "Too many features! Should be less than 300"

    model =  LinearRegression().fit(
        X_train,
        y_train[:, feature_num]
    )
    predictions = model.predict(X_val)
    rmse = mean_squared_error(
        predictions,
        y_val[:, feature_num]
    )**.5
    normalized_rmse = rmse * SCALING_WEIGHTS[feature_num]
    print(f"Property #{feature_num}:    raw RMSE={rmse:.6f}")
    print(f"Property #{feature_num}: scaled RMSE={normalized_rmse:.6f}")
    return round(normalized_rmse, 6)

## Let's try a baseline solution

In [ ]:
def dummy_feature_extractor(X):
    X_new = X.reshape((X.shape[0], -1)) # ravel
    X_new = X_new[:, :300] # pick first 300 features
    return X_new

In [ ]:
 dummy_feature_extractor(X_train).shape

In [ ]:
%%time
total_score = 0
for feature_number in range(3):
  total_score += test_solution(
      dummy_feature_extractor(X_train),
      y_train,
      dummy_feature_extractor(X_val),
      y_val,
      feature_num=feature_number
  )
  print()
total_score /= 3
print('='*16)
print(f"Total score = {total_score:.6f}")

# How to prepare the answer files

In [ ]:
def generate_predictions(X_train, y_train, X_test, feature_num=0):
    assert X_train.shape[-1] <= 300
    assert X_test.shape[-1] <= 300

    model =  LinearRegression().fit(
        X_train,
        y_train[:, feature_num]
    )
    predictions = model.predict(X_test)
    return predictions


## Generate solutions and write to the file
combined = {'ID': np.arange(X_test.shape[0])}

for feature_number in range(3):
    predictions = generate_predictions(
        dummy_feature_extractor(X_train),
        y_train,
        dummy_feature_extractor(X_test),
        feature_num=feature_number
    )

    combined[f'y{feature_number+1}'] = predictions

pd.DataFrame(combined).to_csv('predictions.csv', index=False)

In [ ]:
# load the test dataset
loaded = pd.read_pickle("./test_set/ml_data_onsite_final_test.pickle")
X_test_final = loaded['X']['final_test']


# make final predictions
combined = {'ID': np.arange(X_test_final.shape[0])}

for feature_number in range(3):
    predictions = generate_predictions(
        dummy_feature_extractor(X_train),
        y_train,
        dummy_feature_extractor(X_test_final),
        feature_num=feature_number
    )

    combined[f'y{feature_number+1}'] = predictions

pd.DataFrame(combined).to_csv('final_predictions.csv', index=False)

# 풀이 (Solution)

베이스라인의 `dummy_feature_extractor`(원본 특징 그대로)를 **특징공학**으로 교체한다 — *"feature_extractor 를 개선하라"* 자리.

**모범답안 특징공학**: ① **축순열 대칭 증강**(×6, 라벨 불변) ② **PCA(299)** 차원축소 ③ **배경(bg) 특징**(홈태스크 지식 — 채널3 min − 채널2 max). 셋을 결합 → 검증 Total score ≈ 6.82 → 1.65.

In [ ]:
##########################
# Your code here
##########################

## 평가
최종 특징으로 `test_solution`(속성별 가중 RMSE, **낮을수록 좋음**)을 돌려 검증 점수를 확인한다.

In [ ]:
%%time
total_score = 0
for feature_number in range(3):
    total_score += test_solution(X_train_feat, y_train_symm, X_val_feat, y_val, feature_num=feature_number)
    print()
total_score /= 3
print('='*16)
print(f"Total score = {total_score:.6f}")

## 제출 파일 생성
검증셋 예측을 `submission.csv`(ID, y1, y2, y3)로 저장(테스트 라벨 비공개 → 검증셋으로 채점).

In [ ]:
# 로컬 채점용: 최종 특징(PCA+bg)으로 검증셋(val) 예측 → submission.csv
combined = {'ID': np.arange(X_val.shape[0])}
for _f in range(3):
    combined[f"y{_f+1}"] = generate_predictions(X_train_feat, y_train_symm, X_val_feat, feature_num=_f)
pd.DataFrame(combined).to_csv('submission.csv', index=False)
print("submission.csv (검증셋 예측) 저장:", X_val.shape[0])

## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.csv']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)